# NFL Game Prediction Model - Analysis

This notebook provides analysis and visualization of the NFL prediction model.

## Contents
1. Data Exploration
2. Feature Analysis
3. Model Performance
4. Prediction Examples
5. Feature Importance

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

## 1. Data Exploration

In [ ]:
# Load raw game data
games_df = pd.read_csv('../data/raw_games.csv')
games_df['gameday'] = pd.to_datetime(games_df['gameday'])

print(f"Total games: {len(games_df)}")
print(f"Date range: {games_df['gameday'].min()} to {games_df['gameday'].max()}")
print(f"Seasons: {sorted(games_df['season'].unique())}")

games_df.head()

In [ ]:
# Games per season
games_per_season = games_df.groupby('season').size()

plt.figure(figsize=(12, 6))
games_per_season.plot(kind='bar')
plt.title('Games per Season', fontsize=14, fontweight='bold')
plt.xlabel('Season')
plt.ylabel('Number of Games')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Home team win percentage
games_with_results = games_df[games_df['home_score'].notna() & games_df['away_score'].notna()]
home_wins = (games_with_results['home_score'] > games_with_results['away_score']).sum()
total_games = len(games_with_results)
home_win_pct = home_wins / total_games

print(f"Home team wins: {home_wins}/{total_games} ({100*home_win_pct:.1f}%)")
print(f"This demonstrates the home field advantage in the NFL")

## 2. Feature Analysis

In [ ]:
# Load engineered features
features_df = pd.read_csv('../data/features.csv')

print(f"Total training examples: {len(features_df)}")
print(f"Number of features: {len([col for col in features_df.columns if col not in ['game_id', 'season', 'week', 'home_team', 'away_team', 'gameday', 'home_won', 'score_diff']])}")

features_df.head()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Elo difference
axes[0, 0].hist(features_df['elo_diff'], bins=50, edgecolor='black')
axes[0, 0].set_title('Elo Rating Difference Distribution')
axes[0, 0].set_xlabel('Elo Diff (Home - Away)')
axes[0, 0].axvline(x=0, color='red', linestyle='--', label='Equal teams')

# Recent wins
axes[0, 1].hist(features_df['home_recent_wins'], bins=6, alpha=0.5, label='Home', edgecolor='black')
axes[0, 1].hist(features_df['away_recent_wins'], bins=6, alpha=0.5, label='Away', edgecolor='black')
axes[0, 1].set_title('Recent Wins Distribution (Last 5 Games)')
axes[0, 1].set_xlabel('Wins')
axes[0, 1].legend()

# Season win percentage
axes[0, 2].hist(features_df['home_season_win_pct'], bins=20, alpha=0.5, label='Home', edgecolor='black')
axes[0, 2].hist(features_df['away_season_win_pct'], bins=20, alpha=0.5, label='Away', edgecolor='black')
axes[0, 2].set_title('Season Win Percentage Distribution')
axes[0, 2].set_xlabel('Win %')
axes[0, 2].legend()

# Rest days
axes[1, 0].hist(features_df['home_rest_days'], bins=20, alpha=0.5, label='Home', edgecolor='black')
axes[1, 0].hist(features_df['away_rest_days'], bins=20, alpha=0.5, label='Away', edgecolor='black')
axes[1, 0].set_title('Rest Days Distribution')
axes[1, 0].set_xlabel('Days')
axes[1, 0].legend()

# Point differential
axes[1, 1].hist(features_df['home_recent_diff'], bins=30, alpha=0.5, label='Home', edgecolor='black')
axes[1, 1].hist(features_df['away_recent_diff'], bins=30, alpha=0.5, label='Away', edgecolor='black')
axes[1, 1].set_title('Recent Point Differential')
axes[1, 1].set_xlabel('Point Diff')
axes[1, 1].legend()

# Target distribution
axes[1, 2].bar(['Away Wins', 'Home Wins'], 
               [(features_df['home_won'] == 0).sum(), (features_df['home_won'] == 1).sum()],
               edgecolor='black')
axes[1, 2].set_title('Target Variable Distribution')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Model Performance

In [ ]:
# Load model metadata
import json

try:
    with open('../models/metadata.json', 'r') as f:
        metadata = json.load(f)
    
    print(f"Best Model: {metadata['best_model']}")
    print(f"Training Date: {metadata['timestamp']}")
    print("\nModel Comparison:")
    print("-" * 70)
    
    results_df = pd.DataFrame(metadata['results']).T
    results_df = results_df.sort_values('test_accuracy', ascending=False)
    print(results_df)
    
except FileNotFoundError:
    print("Model metadata not found. Please train models first using: python run_pipeline.py")

In [ ]:
# Visualize model comparison
try:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Accuracy comparison
    results_df[['train_accuracy', 'test_accuracy']].plot(kind='bar', ax=ax1)
    ax1.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Model')
    ax1.legend(['Training', 'Test'])
    ax1.set_xticklabels(results_df.index, rotation=45, ha='right')
    ax1.axhline(y=0.5, color='red', linestyle='--', label='Random guess', alpha=0.5)
    ax1.set_ylim([0.4, 0.8])
    
    # Brier score comparison
    results_df['test_brier'].plot(kind='bar', ax=ax2, color='orange')
    ax2.set_title('Model Brier Score (Lower is Better)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Brier Score')
    ax2.set_xlabel('Model')
    ax2.set_xticklabels(results_df.index, rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
except:
    print("Unable to create visualizations. Ensure models have been trained.")

## 4. Prediction Examples

In [ ]:
from prediction import NFLGamePredictor

# Load predictor
try:
    predictor = NFLGamePredictor()
    
    # Example predictions
    games = [
        {'home_team': 'Kansas City Chiefs', 'away_team': 'Buffalo Bills'},
        {'home_team': 'San Francisco 49ers', 'away_team': 'Dallas Cowboys'},
        {'home_team': 'Philadelphia Eagles', 'away_team': 'New York Giants'},
        {'home_team': 'Baltimore Ravens', 'away_team': 'Cincinnati Bengals'},
        {'home_team': 'Green Bay Packers', 'away_team': 'Chicago Bears'}
    ]
    
    results = predictor.predict_multiple_games(games)
    
    # Display results in a table
    results_data = []
    for r in results:
        results_data.append({
            'Matchup': f"{r['away_team']} @ {r['home_team']}",
            'Predicted Winner': r['predicted_winner'],
            'Confidence': f"{r['confidence']:.1%}",
            'Home Win %': f"{r['home_win_probability']:.1%}",
            'Away Win %': f"{r['away_win_probability']:.1%}"
        })
    
    predictions_df = pd.DataFrame(results_data)
    print(predictions_df.to_string(index=False))
    
except FileNotFoundError:
    print("Model not found. Please train the model first using: python run_pipeline.py")

In [ ]:
# Visualize predictions
try:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    matchups = [f"{r['away_team'][:15]} @\n{r['home_team'][:15]}" for r in results]
    home_probs = [r['home_win_probability'] * 100 for r in results]
    away_probs = [r['away_win_probability'] * 100 for r in results]
    
    x = np.arange(len(matchups))
    width = 0.35
    
    ax.bar(x - width/2, home_probs, width, label='Home Win %', color='#2E7D32')
    ax.bar(x + width/2, away_probs, width, label='Away Win %', color='#C62828')
    
    ax.set_ylabel('Win Probability (%)', fontsize=12)
    ax.set_title('Predicted Win Probabilities', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(matchups, fontsize=9)
    ax.legend()
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim([0, 100])
    
    plt.tight_layout()
    plt.show()
    
except:
    print("Unable to create visualization")

## 5. Feature Importance

In [ ]:
# Load best model and analyze feature importance
try:
    model = joblib.load('../models/best_model.joblib')
    
    if hasattr(model, 'get_feature_importance'):
        importance_df = model.get_feature_importance(top_n=20)
        
        if importance_df is not None:
            plt.figure(figsize=(12, 8))
            plt.barh(range(len(importance_df)), importance_df['importance'])
            plt.yticks(range(len(importance_df)), importance_df['feature'])
            plt.xlabel('Importance', fontsize=12)
            plt.title('Top 20 Most Important Features', fontsize=14, fontweight='bold')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
            
            print("\nTop 20 Features:")
            print(importance_df.to_string(index=False))
        else:
            print(f"Feature importance not available for {model.name}")
    else:
        print("This model type does not support feature importance analysis")
        
except FileNotFoundError:
    print("Model not found. Please train the model first.")

## 6. Custom Analysis

Use the cells below for your own analysis and experimentation.

In [ ]:
# Your code here
